In [31]:
import cv2
import numpy as np
from scipy.signal import butter, filtfilt
from scipy.fft import fft, fftfreq
import collections
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import neurokit2 as nk

class RemotePPG:
    """Бесконтактный rPPG монитор с расчётом ЧСС методами FFT и NeuroKit2"""
    
    def __init__(self, fps=30, buffer_sec=10, hr_update_sec=2.0):
        self.fps = fps
        self.buffer_size = int(fps * buffer_sec)
        self.hr_update_sec = hr_update_sec
        
        # Буфер для сырого хроматического сигнала (G-R)/(G+R)
        self.raw_buffer = collections.deque(maxlen=self.buffer_size)
        
        self.last_hr_time = 0
        self.current_hr_fft = None
        self.current_hr_nk = None
        
        # Каскад для детекции лица
        self.face_cascade = cv2.CascadeClassifier(
            cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
        )

    def _detect_face(self, frame):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.equalizeHist(gray)
        faces = self.face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
        if len(faces) == 0:
            return None
        x, y, w, h = faces[0]
        # ROI: лоб/верхние скулы (наиболее стабильная зона для rPPG)
        roi_h = int(h * 0.3)
        roi_x = x + w // 4
        roi_y = y
        roi_w = w // 2
        return (roi_x, roi_y, roi_w, roi_h), (x, y, w, h)

    def _extract_roi_signal(self, frame, roi):
        """Расчёт (G-R)/(G+R) с последующим усреднением по ROI"""
        x, y, w, h = roi
        if x + w > frame.shape[1] or y + h > frame.shape[0]:
            return None
        
        roi_frame = frame[y:y+h, x:x+w].astype(np.float32)
        b_ch, g_ch, r_ch = cv2.split(roi_frame)  # OpenCV порядок: B, G, R
        
        g_mean = np.mean(g_ch)
        r_mean = np.mean(r_ch)
        return (g_mean - r_mean) / (g_mean + r_mean + 1e-6)

    def _filter_ppg(self, signal):
        """Полосовая фильтрация 0.5–8 Гц"""
        if len(signal) < 30:  # Защита от ошибки filtfilt при малом буфере
            return signal
        b, a = butter(4, [0.5, 8.0], btype='band', fs=self.fps)
        return filtfilt(b, a, signal)

    def hampel_filter(self, signal, window_size=7, threshold=3.0):
        """
        Hampel фильтр для удаления выбросов
        window_size: окно для локальной медианы (нечётное)
        threshold: порог в единицах MAD (median absolute deviation)
        """
        signal = np.array(signal)
        clean_signal = signal.copy()
        
        for i in range(len(signal)):
            # Локальное окно
            start = max(0, i - window_size//2)
            end = min(len(signal), i + window_size//2 + 1)
            window = signal[start:end]
            
            # Медиана и MAD
            median = np.median(window)
            mad = np.median(np.abs(window - median))
            
            # Порог (3.0 ~ 99.7% для нормального распределения)
            if mad > 0:
                if abs(signal[i] - median) > threshold * mad * 1.4826:
                    clean_signal[i] = median  # Заменяем выброс на медиану
        
        return clean_signal

    # def _compute_heart_rate_fft(self, signal):
    #     """Расчёт ЧСС через БПФ с zero-padding для точности"""
    #     signal_clean = np.array([s for s in signal if np.isfinite(s)])
    #     if len(signal_clean) < self.fps * 5:
    #         return None
            
    #     signal_detrended = signal_clean - np.mean(signal_clean)
    #     window = np.hanning(len(signal_detrended))
        
    #     # Zero-padding (в 8 раз) для сглаживания спектра и повышения разрешения
    #     n_pad = len(signal_detrended) * 8
    #     fft_vals = fft(signal_detrended * window, n=n_pad)
    #     freqs = fftfreq(n_pad, 1/self.fps)
        
    #     pos = freqs > 0
    #     freqs_pos, mag = freqs[pos], np.abs(fft_vals[pos])
        
    #     hr_mask = (freqs_pos >= 40/60) & (freqs_pos <= 200/60)
    #     if not np.any(hr_mask):
    #         return None
            
    #     peak_idx = np.argmax(mag[hr_mask])
    #     peak_freq = freqs_pos[hr_mask][peak_idx]
    #     return peak_freq * 60

    def _compute_heart_rate_fft(self, signal):
        """Расчёт ЧСС через БПФ с обрезкой выбросов и zero-padding"""
        signal_clean = np.array([s for s in signal if np.isfinite(s)])
        if len(signal_clean) < self.fps * 5:
            return None
            
        # === ОБРЕЗКА ВЫБРОСОВ ===
        # Ограничиваем сигнал 2-м и 98-м перцентилями (убираем резкие пики/провалы)
        p_low, p_high = np.percentile(signal_clean, [2, 98])
        signal_clean = np.clip(signal_clean, p_low, p_high)

        # Детрендирование и окно Хэннинга
        signal_detrended = signal_clean - np.mean(signal_clean)
        window = np.hanning(len(signal_detrended))
        signal_windowed = signal_detrended * window

        # Zero-padding (в 8 раз) для сглаживания спектра и повышения разрешения
        n_pad = len(signal_detrended) * 8
        fft_vals = fft(signal_windowed, n=n_pad)
        freqs = fftfreq(n_pad, 1/self.fps)

        pos = freqs > 0
        freqs_pos, mag = freqs[pos], np.abs(fft_vals[pos])

        hr_mask = (freqs_pos >= 40/60) & (freqs_pos <= 200/60)
        if not np.any(hr_mask):
            return None
            
        peak_idx = np.argmax(mag[hr_mask])
        peak_freq = freqs_pos[hr_mask][peak_idx]
        return peak_freq * 60

    def _compute_heart_rate_neurokit(self, signal):
        """Расчёт ЧСС через NeuroKit2 (очистка + поиск пиков)"""
        signal_clean = np.array([s for s in signal if np.isfinite(s)])
        if len(signal_clean) < self.fps * 5:
            return None
        try:
            df, _ = nk.ppg_process(signal_clean, sampling_rate=self.fps, show=False)
            hr_series = df["PPG_Rate"].dropna()
            return hr_series.iloc[-1] if len(hr_series) > 0 else None
        except Exception:
            return None

    def process_frame(self, frame):
        result = {'heart_rate_fft': None, 'heart_rate_nk': None, 'face_detected': False, 'roi': None, 'face_rect': None}
        
        det = self._detect_face(frame)
        if det is None:
            return result
            
        roi, face_rect = det
        result['face_detected'] = True
        result['roi'] = roi
        result['face_rect'] = face_rect
        
        sig = self._extract_roi_signal(frame, roi)
        if sig is None or not np.isfinite(sig):
            return result
            
        self.raw_buffer.append(sig)
        
        # Расчёт ЧСС обоими методами каждые 2 секунды
        now = time.time()
        if len(self.raw_buffer) >= self.fps * 10 and (now - self.last_hr_time) >= self.hr_update_sec:
            sig_arr = np.array(list(self.raw_buffer))
            sig_arr = self.hampel_filter(sig_arr, window_size=7, threshold=3.0)
            self.current_hr_fft = self._compute_heart_rate_fft(sig_arr)
            self.current_hr_nk = self._compute_heart_rate_neurokit(sig_arr)
            self.last_hr_time = now
            
        result['heart_rate_fft'] = self.current_hr_fft
        result['heart_rate_nk'] = self.current_hr_nk
        return result

    def draw_roi_on_frame(self, frame, result):
        overlay = frame.copy()
        
        if result['face_rect'] is not None:
            x, y, w, h = result['face_rect']
            cv2.rectangle(overlay, (x, y), (x+w, y+h), (255, 0, 0), 2)
            cv2.putText(overlay, 'Face', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
            
        if result['roi'] is not None:
            x, y, w, h = result['roi']
            h_img, w_img = frame.shape[:2]
            x, y = max(0, x), max(0, y)
            w, h = min(w, w_img - x), min(h, h_img - y)
            if w > 0 and h > 0:
                roi_slice = overlay[y:y+h, x:x+w]
                green_rect = np.full_like(roi_slice, (0, 255, 0), dtype=np.uint8)
                overlay[y:y+h, x:x+w] = cv2.addWeighted(roi_slice, 0.7, green_rect, 0.3, 0)
                cv2.putText(overlay, 'ROI (PPG)', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                
        # Вывод обоих результатов на экран
        y_pos = 50
        if result['heart_rate_fft'] is not None:
            txt = f"FFT HR: {result['heart_rate_fft']:.1f} BPM"
            cv2.putText(frame, txt, (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
            y_pos += 35
            
        if result['heart_rate_nk'] is not None:
            txt = f"NeuroKit HR: {result['heart_rate_nk']:.1f} BPM"
            cv2.putText(frame, txt, (20, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 0), 2)
            
        status = "Face: OK" if result['face_detected'] else "Face: NOT FOUND"
        color = (0, 255, 0) if result['face_detected'] else (0, 0, 255)
        cv2.putText(frame, status, (20, frame.shape[0]-80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        cv2.putText(frame, "Press 'q' to quit", (20, frame.shape[0]-40), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
        
        return cv2.addWeighted(frame, 1, overlay, 0.4, 0)

    def plot_signals(self):
        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(8, 9), dpi=100)
        fig.tight_layout(pad=3.0)
        
        if len(self.raw_buffer) > 10:
            raw = np.array(list(self.raw_buffer))
            ham = self.hampel_filter(raw, window_size=7, threshold=3.0)
            t = np.arange(len(raw)) / self.fps

            # 1. Сырой сигнал
            ax1.plot(t, raw, 'k-', linewidth=0.8, label='Raw (G-R)/(G+R)')
            ax1.plot(t, ham, 'k-', linewidth=0.8, label='Ham (G-R)/(G+R)')
            ax1.set_ylabel('Chrominance Index')
            ax1.set_title('1. Raw Signal')
            ax1.grid(True, alpha=0.3)
            ax1.legend(fontsize=8)
            ax1.set_xlim([0, len(raw)/self.fps])
        else:
            ax1.text(0.5, 0.5, 'Collecting data...', ha='center', va='center', transform=ax1.transAxes)
            ax1.set_title('1. Raw Signal')
            
        # 2. Отфильтрованный сигнал
        if len(self.raw_buffer) >= self.fps * 2:
            filtered = self._filter_ppg(ham)
            t = np.arange(len(filtered)) / self.fps
            ax2.plot(t, filtered, 'b-', linewidth=0.8, label='Filtered (0.5-8 Hz)')
            ax2.set_ylabel('Amplitude (norm.)')
            ax2.set_title('2. Filtered Signal')
            ax2.grid(True, alpha=0.3)
            ax2.legend(fontsize=8)
            ax2.set_xlim([0, len(filtered)/self.fps])
        else:
            ax2.text(0.5, 0.5, 'Filtering...', ha='center', va='center', transform=ax2.transAxes)
            ax2.set_title('2. Filtered Signal')
            
        # 3. Спектр
        if len(self.raw_buffer) >= self.fps * 5:
            detrended = ham - np.mean(ham)
            window = np.hanning(len(detrended))
            n_pad = len(detrended) * 8
            fft_vals = fft(detrended * window, n=n_pad)
            freqs = fftfreq(n_pad, 1/self.fps)
            pos = freqs > 0
            freqs_pos, mag = freqs[pos], np.abs(fft_vals[pos])
            mag = mag / np.max(mag) if np.max(mag) > 0 else mag

            ax3.plot(freqs_pos, mag, 'r-', linewidth=0.8)
            ax3.set_xlabel('Frequency (Hz)')
            ax3.set_ylabel('Norm. Magnitude')
            ax3.set_title('3. Frequency Spectrum')
            ax3.grid(True, alpha=0.3)
            ax3.set_xlim([0, 5])
        else:
            ax3.text(0.5, 0.5, 'Need ~5s for spectrum', ha='center', va='center', transform=ax3.transAxes)
            ax3.set_title('3. Frequency Spectrum')
            
        fig.canvas.draw()
        plot_img = np.asarray(fig.canvas.buffer_rgba())
        plot_img = cv2.cvtColor(plot_img, cv2.COLOR_RGBA2BGR)
        plt.close(fig)
        return plot_img


def main():
    rppg = RemotePPG(fps=30, buffer_sec=10, hr_update_sec=2.0)
    cap = cv2.VideoCapture(0)
    
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    cap.set(cv2.CAP_PROP_FPS, 30)
    
    print("Запуск rPPG (FFT + NeuroKit2). Нажмите 'q' для выхода.")
    print("Советы: равномерный свет, мин. движений, 30-60 см до камеры.")
    
    cv2.namedWindow('Camera - ROI & HR', cv2.WINDOW_NORMAL)
    cv2.namedWindow('Signal Analysis', cv2.WINDOW_NORMAL)
    cv2.resizeWindow('Signal Analysis', 800, 600)

    last_plot_time = 0
    plot_update_interval = 0.5
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
                
            result = rppg.process_frame(frame)
            frame = rppg.draw_roi_on_frame(frame, result)
            
            current_time = time.time()
            if current_time - last_plot_time >= plot_update_interval:
                plot_img = rppg.plot_signals()
                cv2.imshow('Signal Analysis', plot_img)
                last_plot_time = current_time
                
            cv2.imshow('Camera - ROI & HR', frame)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()
        plt.close('all')
        print("Завершено.")

if __name__ == "__main__":
    main()

Запуск rPPG (FFT + NeuroKit2). Нажмите 'q' для выхода.
Советы: равномерный свет, мин. движений, 30-60 см до камеры.


C:\Users\nKovalenko\AppData\Roaming\Python\Python312\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
C:\Users\nKovalenko\AppData\Roaming\Python\Python312\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
C:\Users\nKovalenko\AppData\Roaming\Python\Python312\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
C:\Users\nKovalenko\AppData\Roaming\Python\Python312\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
C:\Users\nKovalenko\AppData\Roaming\Python\Python312\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(
C:\Users\n

Завершено.
